In [ ]:
import os
import chromadb
from datasets import load_dataset
from google import genai

# Local environment configuration
CHROMA_PATH = r"chroma_db"

# Block Execution
raise ValueError("Script already executed ")
# Connect to local ChromaDB instance
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_or_create_collection(name="squad_benchmark")

# Download and load the SQuAD dataset automatically from Hugging Face
print("Downloading SQuAD dataset from Hugging Face...")
raw_dataset = load_dataset("rajpurkar/squad", split="train")

# Connect to the Gemini API Client using Google Cloud Vertex AI
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "gcp-key.json"

ai_client = genai.Client(
    vertexai=True,
    project="info-h512-project-rag-project",
    location="europe-west1"
)

documents = []
metadata = []
ids = []
embeddings = []

# To avoid hitting API quotas or saturating disk memory during testing,
# we index a clean sample slice of 500 unique context passages.
max_passages = 500
seen_contexts = set()

print(f"Processing and embedding the first {max_passages} unique SQuAD contexts...")

for i, entry in enumerate(raw_dataset):
    context_text = entry["context"]
    
    # SQuAD repeats identical context passages for multiple questions; filter for uniqueness
    if context_text in seen_contexts:
        continue
    seen_contexts.add(context_text)
    
    # Store text and unique ID mapping
    documents.append(context_text)
    doc_id = f"SQUAD_{len(seen_contexts)}"
    ids.append(doc_id)
    
    # Extract metadata provided natively by the dataset
    meta = {
        "title": entry["title"],
        "id": entry["id"]
    }
    metadata.append(meta)
    
    # Calculate vector embeddings using Vertex AI text embedding model
    response = ai_client.models.embed_content(
        model="text-embedding-004",
        contents=context_text
    )
    embeddings.append(response.embeddings[0].values)
    
    # Break early once we hit our sample target scale
    if len(seen_contexts) >= max_passages:
        break

# Save the SQuAD text strings and vector coordinates into your local storage
print("Saving SQuAD records to local ChromaDB storage...")
collection.upsert(
    documents=documents,
    embeddings=embeddings,
    metadatas=metadata,
    ids=ids
)
print(f"Successfully indexed {len(documents)} SQuAD passages!")

Processing and embedding the first 500 unique SQuAD contexts...
Saving SQuAD records to local ChromaDB storage...
Successfully indexed 500 SQuAD passages!


In [ ]:
# Connect to local database
CHROMA_PATH = r"chroma_db"
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_or_create_collection(name="squad_benchmark")

# Retrieve all documents physically stored in database
all_db_data = collection.get(include=['documents'])
indexed_contexts = set(all_db_data['documents'])

print(f"Number of documents indexed locally: {len(indexed_contexts)}")

# Load original SQuAD dataset to match questions
print("Searching for associated questions in SQuAD...")
raw_dataset = load_dataset("rajpurkar/squad", split="train")

# Iterate through SQuAD and extract questions for indexed documents
eval_pairs = []
for entry in raw_dataset:
    if entry["context"] in indexed_contexts:
        eval_pairs.append({
            "question": entry["question"],
            "expected_answer": entry["answers"]["text"][0] if entry["answers"]["text"] else "N/A",
            "context": entry["context"]
        })
    
    # Stop once we have a good sample of questions for testing
    if len(eval_pairs) >= 25:
        break

# Display the questions for testing
print("\n=== Questions / Answers ===\n")
for idx, pair in enumerate(eval_pairs, 1):
    print(f"Question {idx}: {pair['question']}")
    print(f"Expected answer: {pair['expected_answer']}")
    print("-" * 50)

Number of documents indexed locally: 500
Searching for associated questions in SQuAD...

=== TEST QUESTIONS TO USE IN YOUR SWAGGER ===

Question 1: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
Expected answer: Saint Bernadette Soubirous
--------------------------------------------------
Question 2: What is in front of the Notre Dame Main Building?
Expected answer: a copper statue of Christ
--------------------------------------------------
Question 3: The Basilica of the Sacred heart at Notre Dame is beside to which structure?
Expected answer: the Main Building
--------------------------------------------------
Question 4: What is the Grotto at Notre Dame?
Expected answer: a Marian place of prayer and reflection
--------------------------------------------------
Question 5: What sits on top of the Main Building at Notre Dame?
Expected answer: a golden statue of the Virgin Mary
--------------------------------------------------
Question 6: When did the Scholas